# FiliSenti Benchmark: jjjardev models vs public Filipino sentiment classifiers

This notebook benchmarks **jjjardev/tagasenti_model**, **jjjardev/filisenti** and
**jjjardev/hilisenti-v1-model** against the
public, loadable Filipino/Tagalog sentiment classifiers on Hugging Face, verified by a
full API sweep (searches: tagalog, filipino, filisenti, tagasenti, senti, shopee, lazada,
sentiment tagalog/filipino, plus `filter=tl` and `filter=fil` text-classification listings).

**Benchmarkable roster found (14 verified + 1 optional LoRA):**

| # | Model | Type | Labels |
|---|-------|------|--------|
| 1 | jjjardev/tagasenti_model | XLM-R large 3-class | explicit Neg/Neu/Pos (yours) |
| 2 | jjjardev/filisenti | XLM-R large 3-class | explicit Neg/Neu/Pos (yours) |
| 3 | jjjardev/hilisenti-v1-model | XLM-R 3-class | explicit Neg/Neu/Pos (yours, Hiligaynon) |
| 4 | dost-asti/RoBERTa-tl-sentiment-analysis | RoBERTa base 3-class | LABEL_0/1/2, inferred by probe |
| 5 | raphgonda/FilipinoShopping | RoBERTa base 3-class | explicit, 400k Shopee reviews |
| 6 | magixxixx/roberta-shopee-sentiment-gadgets | RoBERTa base 3-class | explicit NEG/Neu/POS |
| 7 | sdbrgo/roberta-tagalog-sentiment-multiclass-classifier | RoBERTa base 3-class | LABEL_0/1/2, inferred by probe |
| 8 | immortalizing-legacies/immortalizing-legacies-sentiment | XLM-R 3-class | explicit Neg/Neu/Pos |
| 9 | MENG21/stud-fac-eval-bert-base-uncased_v2 | BERT, tl/en eval comments | inferred by probe |
| 10 | scaredmeow/xlnet-tagalog-sentiment-shopee | XLNet binary | explicit neg/pos |
| 11 | scaredmeow/roberta-tagalog-sentiment-shopee | RoBERTa binary | inferred by probe |
| 12 | The-Thesis-Gods/lazada-xlm-roberta-sentiment | XLM-R binary | explicit neg/pos |
| 13 | cardiffnlp/twitter-xlm-roberta-base-sentiment | XLM-R base 3-class | explicit (multilingual baseline) |
| 14 | tabularisai/multilingual-sentiment-analysis | DistilBERT 5-class | explicit, collapsed 5 to 3 |
| 15 | letijo03/shopee-taglish-sentiment-analysis | XLM-R base + LoRA | optional cell, needs peft |

**Excluded by the sweep, and why:**
- `jpcurada/nlp-cdk-filipino-sentiment-analysis`: **gated**, needs login approval, cannot run anonymously.
- `ADDUemmanaguado/xlmr-filipino-sentiment` and `Hax/filipino-text-version1`: repos dead (entry not found).
- `scaredmeow/*-sentiment-hatespeech`: trained on hate speech datasets, wrong task despite the name.
- `convaiinnovations/laya-multilingual`: guardrail/routing decision model, not sentiment.
- `jcblaise/*`, `GKLMIP/*`, `scaredmeow/xlnet-tagalog-base`: pretrained language models, no sentiment head.
- Hate speech / profanity / fake news / NER / NLI models: wrong task.
- Tagalog-tuned LLMs (Llama, SealLM 7B+): possible via prompting but a different benchmark class, too heavy for a free T4 alongside 355M classifiers.

**Test sets (7, including one contamination probe):**

| # | Set | n | Kind | Source | Why it earns a slot |
|---|-----|---|------|--------|---------------------|
| 1 | tagasenti_test | 2000 | 3-class | jjjardev/tagasenti | Tagalog, your label schema |
| 2 | hilisenti_test | 2000 | 3-class | jjjardev/hilisenti-v1 | Hiligaynon |
| 3 | shopee_binary | 1000 | binary | scaredmeow/shopee-reviews-tl-binary | Taglish e-commerce, star-derived binary |
| 4 | mteb_shopee_3class | 2000 | 3-class | mteb/FilipinoShopeeReviewsClassification | star-rating neutral, the hard neutral |
| 5 | taglish_annotated_3class | 2000 | 3-class | letijo03/shopee-sentiment | HUMAN-annotated neutral on 50k Taglish Shopee comments, the exact axis where every model collapsed on set 4 |
| 6 | lazada_stars_3class | 1001 | 3-class | EricEchemane repo (packaged by SEACrowd) | second e-commerce platform, star-collapsed |
| 7 | gadgets_train_probe | 1000 | binary | magixxixx 40k gadget reviews | contamination probe: this is magixxixx's own TRAINING data, zero-shot for everyone else |

**Test sets considered and rejected (so nobody asks):**
- `SEACrowd/shopee_reviews_tagalog`: repack of scaredmeow/shopee-reviews-tl-stars, same reviews as sets 3/4, zero new signal.
- `SEACrowd/unisent`: word-level sentiment lexicon, single words are not reviews.
- `lllaurenceee/Shopee_Bicycle_Reviews`: 23k of 30k rows have null comments, 91 percent are 5-star, after filtering the negative class is ~100 rows, too small to score honestly.
- `kornwtp/lazada-review-fil-classification`: 84 KB parquet, same 1k Lazada file as set 6.
- `jcblaise/dengue_filipino`, `fake_news_filipino`, hate speech and sarcasm datasets: real Filipino classification data but the label space is not sentiment, a sentiment head cannot be scored on them without changing the task.

**Protocol:** every model is scored on the same test sets, with no fine-tuning on the test rows
in this run. The TagaSenti and Hiligaynon columns are in-domain for the jjjardev models, so read
those as held-out scores, not zero-shot. Models with undocumented output labels get their label
order inferred automatically from probe sentences. Binary models are scored only on the binary
leaderboard (on binary test sets, 3-class models choose between negative and positive using
renormalized probabilities). `gadgets_train_probe` is a contamination check, not a scoring set.
Runtime scales with `SAMPLE_CAP` below; defaults fit a free Colab T4.

In [1]:
!pip -q install "transformers>=4.44,<5" sentencepiece protobuf datasets scikit-learn peft accelerate pandas matplotlib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import numpy as np, pandas as pd, torch, gc
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 128
BATCH = 32
SAMPLE_CAP = 2000   # per dataset; lower to 500 for a fast smoke test
SEED = 42

# mapping: "explicit" = id2label already meaningful, "probe" = infer from sentences,
# dict = hardcoded rename of model labels to negative/neutral/positive
MODELS = [
    dict(id="jjjardev/tagasenti_model",                       kind="3class", mapping="explicit", note="yours"),
    dict(id="jjjardev/filisenti",                             kind="3class", mapping="explicit", note="yours"),
    dict(id="jjjardev/hilisenti-v1-model",                    kind="3class", mapping="explicit", note="yours, Hiligaynon"),
    dict(id="dost-asti/RoBERTa-tl-sentiment-analysis",        kind="3class", mapping="probe",    note="DOST-ASTI (iTANONG-DS)"),
    dict(id="raphgonda/FilipinoShopping",                     kind="3class", mapping="explicit", note="400k Shopee, TF weights only"),
    dict(id="magixxixx/roberta-shopee-sentiment-gadgets",     kind="3class", mapping="explicit", note="Shopee gadgets"),
    dict(id="sdbrgo/roberta-tagalog-sentiment-multiclass-classifier", kind="3class", mapping="probe", note="synthetic Taglish"),
    dict(id="immortalizing-legacies/immortalizing-legacies-sentiment", kind="3class", mapping="explicit", note=""),
    dict(id="MENG21/stud-fac-eval-bert-base-uncased_v2",      kind="3class", mapping="probe",    note="student-faculty eval"),
    dict(id="scaredmeow/xlnet-tagalog-sentiment-shopee",      kind="binary", mapping="explicit", note="Shopee tl-binary"),
    dict(id="scaredmeow/roberta-tagalog-sentiment-shopee",    kind="binary", mapping="probe",    note="Shopee tl-binary"),
    dict(id="The-Thesis-Gods/lazada-xlm-roberta-sentiment",   kind="binary", mapping="explicit", note="Lazada AutoTrain"),
    dict(id="cardiffnlp/twitter-xlm-roberta-base-sentiment",  kind="3class", mapping="explicit", note="multilingual baseline"),
    dict(id="tabularisai/multilingual-sentiment-analysis",    kind="3class", mapping="explicit", note="5-class collapsed"),
]
print(f"{len(MODELS)} models registered, device = {DEVICE}")

14 models registered, device = cuda


In [3]:
def norm3(df, text_col="sentence", label_col="label"):
    """Standardize to columns text (str) and label in {negative, neutral, positive}."""
    m = {0: "negative", 1: "neutral", 2: "positive"}
    out = df.rename(columns={text_col: "text", label_col: "label"})[["text", "label"]].dropna()
    out["text"] = out["text"].astype(str)
    out["label"] = out["label"].map(m) if out["label"].dtype != object else out["label"].str.lower()
    return out.dropna().reset_index(drop=True)

datasets = {}   # name -> (dataframe, kind)

# 1) jjjardev/tagasenti : single CSV, reconstruct the stratified 80/10/10 seed-42 test split
#    (best-effort approximation of the split used in the tagasenti_model card)
raw = pd.read_csv("https://huggingface.co/datasets/jjjardev/tagasenti/resolve/main/tagasenti_dataset.csv")
rest, test = train_test_split(raw, test_size=0.2, random_state=SEED, stratify=raw["label"])
_, tag_test = train_test_split(test, test_size=0.5, random_state=SEED, stratify=test["label"])
tag_test = norm3(tag_test)
if len(tag_test) > SAMPLE_CAP:
    tag_test, _ = train_test_split(tag_test, train_size=SAMPLE_CAP, random_state=SEED, stratify=tag_test["label"])
datasets["tagasenti_test"] = (tag_test, "3class")

# 2) jjjardev/hilisenti-v1 : ready test split, Hiligaynon
hili = norm3(pd.read_csv("https://huggingface.co/datasets/jjjardev/hilisenti-v1/resolve/main/test.csv"))
if len(hili) > SAMPLE_CAP:
    hili, _ = train_test_split(hili, train_size=SAMPLE_CAP, random_state=SEED, stratify=hili["label"])
datasets["hilisenti_test"] = (hili, "3class")

# 3) scaredmeow/shopee-reviews-tl-binary : external Taglish Shopee reviews, binary
bin_df = pd.read_csv("https://huggingface.co/datasets/scaredmeow/shopee-reviews-tl-binary/resolve/main/test.csv")
bin_df = bin_df.rename(columns={"text": "text", "label": "label"})[["text", "label"]].dropna()
bin_df["text"] = bin_df["text"].astype(str)
bin_df["label"] = bin_df["label"].map({0: "negative", 1: "positive"})
bin_df = bin_df.dropna().reset_index(drop=True)
if len(bin_df) > SAMPLE_CAP // 2:
    bin_df, _ = train_test_split(bin_df, train_size=SAMPLE_CAP // 2, random_state=SEED, stratify=bin_df["label"])
datasets["shopee_binary"] = (bin_df, "binary")

# 4) mteb/FilipinoShopeeReviewsClassification : external MTEB test split
mteb = pd.read_parquet("https://huggingface.co/datasets/mteb/FilipinoShopeeReviewsClassification/resolve/main/data/test-00000-of-00001.parquet")
star_map = {0: "negative", 1: "negative", 2: "neutral", 3: "positive", 4: "positive"}
mteb = mteb.rename(columns={"text": "text", "label": "label"})[["text", "label"]].dropna()
mteb["text"] = mteb["text"].astype(str)
mteb["label"] = mteb["label"].map(star_map)
mteb = mteb.dropna().reset_index(drop=True)
if len(mteb) > SAMPLE_CAP:
    mteb, _ = train_test_split(mteb, train_size=SAMPLE_CAP, random_state=SEED, stratify=mteb["label"])
datasets["mteb_shopee_3class"] = (mteb, "3class")

# 5) letijo03/shopee-sentiment : 50k Taglish Shopee comments with HUMAN annotation labels
#    (Positive 40744 / Neutral 2159 / Neutral (Contradiction) 2615 / Negative 4772).
#    The two neutral flavors are merged. This is the only set where gold neutral comes
#    from an annotator, not from a star rating, which is exactly the axis where every
#    model collapsed on mteb_shopee_3class. NOTE: letijo03's own LoRA model was trained
#    on this file, so treat its score here as in-domain, not zero-shot.
taglish = pd.read_csv("https://huggingface.co/datasets/letijo03/shopee-sentiment/resolve/main/shopee_comment_datasets_annotated.csv")
taglish = taglish.rename(columns={"Comment": "text", "label": "label"})[["text", "label"]].dropna()
taglish["text"] = taglish["text"].astype(str)
taglish["label"] = taglish["label"].replace({"Neutral (Contradiction)": "Neutral"}).str.lower()
taglish = taglish.dropna().reset_index(drop=True)
if len(taglish) > SAMPLE_CAP:
    taglish, _ = train_test_split(taglish, train_size=SAMPLE_CAP, random_state=SEED, stratify=taglish["label"])
datasets["taglish_annotated_3class"] = (taglish, "3class")

# 6) Lazada reviews, 1-5 stars, source file of SEACrowd/lazada_review_filipino
#    (EricEchemane/Filipino-Tagalog-Product-Reviews-Sentiment-Analysis).
#    Stars collapse like the MTEB set: 1-2 negative, 3 neutral, 4-5 positive.
#    n=1001, under the cap, so every row is used. The-Thesis-Gods/lazada was trained on
#    Lazada reviews (likely this exact file), treat its score here as home domain.
laz = pd.read_json("https://raw.githubusercontent.com/EricEchemane/Filipino-Tagalog-Product-Reviews-Sentiment-Analysis/main/data/reviews.json")
laz = laz.rename(columns={"review": "text", "rating": "label"})[["text", "label"]].dropna()
laz["text"] = laz["text"].astype(str)
laz["label"] = pd.to_numeric(laz["label"], errors="coerce").map(
    lambda r: "negative" if r <= 2 else ("neutral" if r == 3 else "positive"))
laz = laz.dropna().reset_index(drop=True)
datasets["lazada_stars_3class"] = (laz, "3class")

# 7) magixxixx 40k computer-category Shopee reviews, binary 0/1.
#    CONTAMINATION PROBE ONLY: this is the training data of
#    magixxixx/roberta-shopee-sentiment-gadgets, so that model's number here is a
#    train-set accuracy check (expect near 1.0). For the other 13 models this is a
#    genuine unseen gadgets-domain set.
gad = pd.read_csv("https://huggingface.co/datasets/magixxixx/shopee-product-reviews-on-computer-category/resolve/main/combined-dataset-shuffled-cased-40K.csv")
gad = gad.rename(columns={"text": "text", "sentiment": "label"})[["text", "label"]].dropna()
gad["text"] = gad["text"].astype(str)
gad["label"] = gad["label"].map({0: "negative", 1: "positive"})
gad = gad.dropna().reset_index(drop=True)
if len(gad) > SAMPLE_CAP // 2:
    gad, _ = train_test_split(gad, train_size=SAMPLE_CAP // 2, random_state=SEED, stratify=gad["label"])
datasets["gadgets_train_probe"] = (gad, "binary")

for name, (df, kind) in datasets.items():
    print(f"{name:16s} {kind:7s} n={len(df):5d}  {df['label'].value_counts().to_dict()}")

# sanity check: print 1 example per label of each 3-class set to verify the 0/1/2 order
for name in ["tagasenti_test", "hilisenti_test", "mteb_shopee_3class",
             "taglish_annotated_3class", "lazada_stars_3class"]:
    print(f"\n--- {name} label sanity ---")
    for lab in ["negative", "neutral", "positive"]:
        ex = datasets[name][0][datasets[name][0]["label"] == lab]["text"].iloc[0]
        print(f"  {lab:9s}: {ex[:100]}")

tagasenti_test   3class  n= 2000  {'negative': 743, 'neutral': 653, 'positive': 604}
hilisenti_test   3class  n= 2000  {'negative': 738, 'positive': 697, 'neutral': 565}
shopee_binary    binary  n= 1000  {'positive': 500, 'negative': 500}
mteb_shopee_3class 3class  n= 2000  {'negative': 800, 'positive': 800, 'neutral': 400}
taglish_annotated_3class 3class  n= 2000  {'positive': 1620, 'negative': 190, 'neutral': 190}
lazada_stars_3class 3class  n= 1001  {'positive': 498, 'negative': 380, 'neutral': 123}
gadgets_train_probe binary  n= 1000  {'negative': 500, 'positive': 500}

--- tagasenti_test label sanity ---
  negative : Nahulog ang mga damit mula sa sampayan dahil sa lakas ng hangin at umulan pa.
  neutral  : Ang pagiging alilang-kanin ay naging bahagi ng buhay ng mga naulila noong panahon ng digmaan.
  positive : makapal yung tela pwede panlamig sulit sa presyo great quality

--- hilisenti_test label sanity ---
  negative : Suno kay Lacson, ang tubig para sa 1 2 million peso bulk wa

In [4]:
POS_PROBES = [
    "Sobrang ganda ng quality, sulit na sulit ang bayad! Mahal na mahal ko ito.",
    "Ang sarap ng pagkain dito at napakabait ng staff, babalik balikan ko talaga.",
]
NEG_PROBES = [
    "Ang pangit ng serbisyo, sobrang tagal at ang rude ng staff. Ayaw ko na dito.",
    "Basag ang dumating at ayaw palitan ng seller. Hindi na ako bibili ulit, nakakainis.",
]
NEU_PROBES = [
    "Nakarating na ang package kahapon ng hapon sa aming bahay.",
    "Ang meeting ay gaganapin sa Lunes alas-tres ng hapon sa opisina.",
]

@torch.no_grad()
def predict_probs(texts, tok, model, batch_size=BATCH):
    probs = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tok(batch, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
            logits = model(**enc).logits
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.vstack(probs)

def infer_label_order(tok, model, raw_id2label):
    """For models whose config only says LABEL_0/1/2: score probe sentences per class and
    assign each raw index the sentiment whose probes it scores highest."""
    probes = {"negative": NEG_PROBES, "neutral": NEU_PROBES, "positive": POS_PROBES}
    all_texts = [t for ts in probes.values() for t in ts]
    p = predict_probs(all_texts, tok, model)
    n_probe = len(POS_PROBES)
    inferred = {}
    for idx, raw in raw_id2label.items():
        scores = {}
        for senti, ts in probes.items():
            rows = [i for i, t in enumerate(all_texts) if t in ts]
            scores[senti] = float(p[rows, idx].mean())
        inferred[idx] = max(scores, key=scores.get)
    return inferred

# how to fold each raw class distribution into the common negative/neutral/positive space
def build_fold(model_cfg, mapping_kind, n_labels):
    if mapping_kind == "explicit" and n_labels == 5:
        # tabularisai: Very Negative, Negative, Neutral, Positive, Very Positive
        return lambda P: np.stack([P[:, 0] + P[:, 1], P[:, 2], P[:, 3] + P[:, 4]], axis=1)
    return None  # handled per-model below

def make_explicit_fold3(raw_lower):
    """Column reorder for models whose id2label is already an explicit 3-class mapping.
    Indexes numpy with a LIST of column ids, never a dict (a dict raises IndexError)."""
    cols = {i: k for i, k in raw_lower.items()
            if k in ("negative", "neutral", "positive",
                     "negative-sentiment", "neutral-sentiment", "positive-sentiment")}
    inv = {0: [i for i, k in cols.items() if "neg" in k][0],
           1: [i for i, k in cols.items() if "neu" in k][0],
           2: [i for i, k in cols.items() if "pos" in k][0]}
    return lambda P: P[:, [inv[0], inv[1], inv[2]]]

# self-test, guards the dict-indexing bug that once silently skipped 7 models
_t = make_explicit_fold3({0: "negative", 1: "neutral", 2: "positive"})
_d = np.array([[0.1, 0.2, 0.7], [0.8, 0.1, 0.1]])
assert np.allclose(_t(_d), _d), "identity order must pass through"
_r = make_explicit_fold3({0: "positive", 1: "neutral", 2: "negative"})
assert np.allclose(_r(_d), _d[:, [2, 1, 0]]), "reversed order must flip columns"
del _t, _r, _d
print("fold builder self-test passed")

fold builder self-test passed


In [5]:
results = []
loaded_probe_maps = {}

for spec in MODELS:
    mid = spec["id"]
    print(f"\n{'='*70}\nLOADING {mid} ({spec['note']})")
    try:
        tok = AutoTokenizer.from_pretrained(mid)
        try:
            model = AutoModelForSequenceClassification.from_pretrained(mid).to(DEVICE)
        except OSError as e:
            # some repos ship only TensorFlow weights (tf_model.h5), e.g. raphgonda/FilipinoShopping
            if "TensorFlow weights" not in str(e) and "from_tf" not in str(e):
                raise
            print("  repo has only TF weights, retrying with from_tf=True (needs tensorflow, preinstalled on Colab)")
            model = AutoModelForSequenceClassification.from_pretrained(mid, from_tf=True).to(DEVICE)
    except Exception as e:
        import traceback
        print(f"  SKIP, failed to load. Full error below (report this if it is a tokenizer/model issue):")
        traceback.print_exc()
        continue
    try:
        n_lab = model.config.num_labels
        id2label = model.config.id2label or {i: f"LABEL_{i}" for i in range(n_lab)}
        raw_names = [id2label[i] for i in range(n_lab)]
        undocumented = any(str(r).startswith("LABEL_") for r in raw_names)

        if spec["mapping"] == "probe" or undocumented:
            label_map = infer_label_order(tok, model, id2label)   # raw idx -> sentiment
            loaded_probe_maps[mid] = label_map
            print(f"  inferred label order: {label_map}")
            fold = lambda P, lm=label_map, n=n_lab: (
                np.stack([P[:, [i for i in lm if lm[i] == 'negative']].sum(1),
                          P[:, [i for i in lm if lm[i] == 'neutral']].sum(1) if n == 3 else np.zeros(len(P)),
                          P[:, [i for i in lm if lm[i] == 'positive']].sum(1)], axis=1))
        else:
            raw_lower = {i: str(id2label[i]).lower() for i in range(n_lab)}
            if n_lab == 5:
                fold = build_fold(None, "explicit", 5)
            elif n_lab == 3:
                fold = make_explicit_fold3(raw_lower)
            else:  # binary
                inv = [[i for i, k in raw_lower.items() if "neg" in k][0], [i for i, k in raw_lower.items() if "pos" in k][0]]
                fold = lambda P, inv=inv: P[:, inv]

        for dname, (df, dkind) in datasets.items():
            if dkind == "binary" and spec["kind"] != "binary":
                pass                      # 3-class model on binary set: allowed, see below
            texts = df["text"].tolist()
            gold = df["label"].values
            P_raw = predict_probs(texts, tok, model)
            P = fold(P_raw)
            if spec["kind"] == "binary" and P.shape[1] == 2:
                if dkind != "binary":
                    continue              # binary models only compete on binary sets
                pred = np.where(P.argmax(1) == 0, "negative", "positive")
                labels_eval = ["negative", "positive"]
            else:
                if dkind == "binary":
                    Pb = P[:, [0, 2]]; Pb = Pb / Pb.sum(1, keepdims=True)   # renorm neg vs pos
                    pred = np.where(Pb.argmax(1) == 0, "negative", "positive")
                    labels_eval = ["negative", "positive"]
                else:
                    pred = np.array(["negative", "neutral", "positive"])[P.argmax(1)]
                    labels_eval = ["negative", "neutral", "positive"]
            mask = np.isin(gold, labels_eval)
            g, p2 = gold[mask], pred[mask]
            acc = accuracy_score(g, p2)
            f1m = f1_score(g, p2, labels=labels_eval, average="macro", zero_division=0)
            rec = recall_score(g, p2, labels=labels_eval, average=None, zero_division=0)
            results.append(dict(model=mid, note=spec["note"], dataset=dname, n=int(mask.sum()),
                                accuracy=round(acc, 4), macro_f1=round(f1m, 4),
                                **{f"recall_{l}": round(r, 4) for l, r in zip(labels_eval, rec)}))
            print(f"  {dname:16s} acc={acc:.4f}  macroF1={f1m:.4f}")
    except Exception as e:
        print(f"  ERROR during eval of {mid}: {e}")
    finally:
        del model, tok
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

res = pd.DataFrame(results)
res.to_csv("benchmark_results.csv", index=False)
print("\nProbe-inferred label maps (keep these, they document each model's raw order):")
for k, v in loaded_probe_maps.items():
    print(f"  {k}: {v}")
res


LOADING jjjardev/tagasenti_model (yours)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/922 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

... (92 outputs left out)

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

  tagasenti_test   acc=0.5430  macroF1=0.5403
  hilisenti_test   acc=0.5180  macroF1=0.5044
  shopee_binary    acc=0.8070  macroF1=0.8070
  mteb_shopee_3class acc=0.5605  macroF1=0.4727
  taglish_annotated_3class acc=0.5945  macroF1=0.4504
  lazada_stars_3class acc=0.6154  macroF1=0.4930
  gadgets_train_probe acc=0.8450  macroF1=0.8450

Probe-inferred label maps (keep these, they document each model's raw order):
  dost-asti/RoBERTa-tl-sentiment-analysis: {0: 'negative', 1: 'positive', 2: 'neutral'}
  sdbrgo/roberta-tagalog-sentiment-multiclass-classifier: {0: 'negative', 1: 'neutral', 2: 'positive'}
  MENG21/stud-fac-eval-bert-base-uncased_v2: {0: 'negative', 1: 'neutral'}
  scaredmeow/roberta-tagalog-sentiment-shopee: {0: 'negative', 1: 'positive'}


,model,note,dataset,n,accuracy,macro_f1,recall_negative,recall_neutral,recall_positive
0,jjjardev/tagasenti_model,yours,tagasenti_test,2000,0.8475,0.8475,0.8439,0.8515,0.8477
1,jjjardev/tagasenti_model,yours,hilisenti_test,2000,0.5870,0.5850,0.6111,0.5805,0.5667
2,jjjardev/tagasenti_model,yours,shopee_binary,1000,0.8780,0.8768,0.7780,NaN,0.9780
3,jjjardev/tagasenti_model,yours,mteb_shopee_3class,2000,0.6535,0.5665,0.7850,0.1825,0.7575
4,jjjardev/tagasenti_model,yours,taglish_annotated_3class,2000,0.7870,0.6338,0.6263,0.5895,0.8290
...,...,...,...,...,...,...,...,...,...
76,tabularisai/multilingual-sentiment-analysis,5-class collapsed,shopee_binary,1000,0.8070,0.8070,0.7920,NaN,0.8220
77,tabularisai/multilingual-sentiment-analysis,5-class collapsed,mteb_shopee_3class,2000,0.5605,0.4727,0.7512,0.1175,0.5913
78,tabularisai/multilingual-sentiment-analysis,5-class collapsed,taglish_annotated_3class,2000,0.5945,0.4504,0.6421,0.3316,0.6198
79,tabularisai/multilingual-sentiment-analysis,5-class collapsed,lazada_stars_3class,1001,0.6154,0.4930,0.7053,0.0976,0.6747


In [6]:
# Leaderboards
assert not res.empty, "No results collected, inspect the SKIP/ERROR blocks above before continuing"
for dname in datasets:
    sub = res[res["dataset"] == dname].sort_values("macro_f1", ascending=False)
    print(f"\n{'='*70}\nLEADERBOARD, {dname} (macro F1)")
    print(sub[["model", "note", "accuracy", "macro_f1"]].to_string(index=False))

# Head-to-head summary: average macro F1 across the six scoring sets only.
# gadgets_train_probe is a contamination check, not a scoring set.
scoring_sets = [s for s in datasets if s != "gadgets_train_probe"]
three_class_ids = [m["id"] for m in MODELS if m["kind"] == "3class"]
summary = (res[res["dataset"].isin(scoring_sets) & res["model"].isin(three_class_ids)]
           .groupby(["model", "note"])["macro_f1"]
           .agg(["mean", "count"]).sort_values("mean", ascending=False))
print("\n" + "=" * 70 + "\nOVERALL (mean macro F1 across the six scoring sets)")
print(summary.to_string())


LEADERBOARD, tagasenti_test (macro F1)
                                                  model                   note  accuracy  macro_f1
                                     jjjardev/filisenti                  yours    0.9325    0.9327
                               jjjardev/tagasenti_model                  yours    0.8475    0.8475
                            jjjardev/hilisenti-v1-model      yours, Hiligaynon    0.6430    0.6454
          cardiffnlp/twitter-xlm-roberta-base-sentiment  multilingual baseline    0.6200    0.6152
immortalizing-legacies/immortalizing-legacies-sentiment                           0.6045    0.5880
            tabularisai/multilingual-sentiment-analysis      5-class collapsed    0.5430    0.5403
                dost-asti/RoBERTa-tl-sentiment-analysis DOST-ASTI (iTANONG-DS)    0.5280    0.4926
 sdbrgo/roberta-tagalog-sentiment-multiclass-classifier      synthetic Taglish    0.4930    0.4192
            scaredmeow/roberta-tagalog-sentiment-shopee       Shopee 

In [7]:
# DOST-ASTI 6-permutation upper bound (recommended before publishing)
# Their config only documents LABEL_0/1/2. This scores them under EVERY possible label
# order, so nobody can claim the benchmark guessed their outputs wrong.
import itertools
PERM_MODEL = "dost-asti/RoBERTa-tl-sentiment-analysis"
print("Computing", PERM_MODEL, "under all 6 label permutations on tagasenti_test ...")
tok = AutoTokenizer.from_pretrained(PERM_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(PERM_MODEL).to(DEVICE)
df_perm, _ = datasets["tagasenti_test"]
P_perm = predict_probs(df_perm["text"].tolist(), tok, model)
gold_perm = df_perm["label"].values
CLASSES = ["negative", "neutral", "positive"]
perm_rows = []
for perm in itertools.permutations(range(3)):
    pred = np.array(CLASSES)[P_perm[:, list(perm)].argmax(1)]
    perm_rows.append(dict(label_order=str(perm),
                          accuracy=round(accuracy_score(gold_perm, pred), 4),
                          macro_f1=round(f1_score(gold_perm, pred, labels=CLASSES, average="macro", zero_division=0), 4)))
perm_df = pd.DataFrame(perm_rows).sort_values("macro_f1", ascending=False)
print(perm_df.to_string(index=False))
print(f"\nBest achievable DOST mapping: {perm_df.iloc[0]['macro_f1']} macro F1. "
      f"If this stays far below your models, the comparison is label-order-proof.")
del model, tok, P_perm
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

Computing dost-asti/RoBERTa-tl-sentiment-analysis under all 6 label permutations on tagasenti_test ...
label_order  accuracy  macro_f1
  (0, 2, 1)    0.5280    0.4926
  (0, 1, 2)    0.3915    0.3426
  (2, 0, 1)    0.3745    0.3329
  (1, 2, 0)    0.2340    0.2373
  (1, 0, 2)    0.2480    0.2198
  (2, 1, 0)    0.2240    0.2004

Best achievable DOST mapping: 0.4926 macro F1. If this stays far below your models, the comparison is label-order-proof.


In [ ]:
import matplotlib.pyplot as plt

scoring_sets = ["tagasenti_test", "hilisenti_test", "shopee_binary",
                "mteb_shopee_3class", "taglish_annotated_3class", "lazada_stars_3class"]
display = {
    "tagasenti_test": "TagaSenti",
    "hilisenti_test": "Hiligaynon",
    "shopee_binary": "Shopee binary",
    "mteb_shopee_3class": "MTEB Shopee",
    "taglish_annotated_3class": "Taglish annotated",
    "lazada_stars_3class": "Lazada stars",
}
three_class_models = [
    "jjjardev/filisenti",
    "jjjardev/hilisenti-v1-model",
    "jjjardev/tagasenti_model",
    "immortalizing-legacies/immortalizing-legacies-sentiment",
    "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "tabularisai/multilingual-sentiment-analysis",
    "dost-asti/RoBERTa-tl-sentiment-analysis",
    "sdbrgo/roberta-tagalog-sentiment-multiclass-classifier",
    "magixxixx/roberta-shopee-sentiment-gadgets",
    "MENG21/stud-fac-eval-bert-base-uncased_v2",
]

pivot = (res[res["model"].isin(three_class_models)]
         .pivot_table(index="model", columns="dataset", values="macro_f1")[scoring_sets]
         .rename(columns=display))
pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(12, 7.5), constrained_layout=True)
pivot.plot(kind="barh", ax=ax, width=0.78,
           color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"])
ax.set_xlabel("Macro F1")
ax.set_ylabel("")
ax.set_title("Filipino sentiment benchmark: macro F1 by test set", fontsize=14, pad=30)
ax.text(0.5, 1.015, "Six scoring sets, ten 3-class models. The gadgets training probe is excluded.",
        transform=ax.transAxes, ha="center", va="bottom", fontsize=9, color="#444444")
ax.set_xlim(0, 1.0)
ax.grid(axis="x", alpha=0.25)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.legend(loc="lower right", fontsize=8, ncol=2, frameon=True)
ax.tick_params(axis="y", labelsize=9)
plt.savefig("benchmark_chart.png", dpi=150, facecolor="white")
plt.show()

yours = res[(res["model"] == "jjjardev/tagasenti_model")]
print(yours.to_string(index=False))

## Notes and caveats

- **TagaSenti test split is a reconstruction.** The Hub CSV is one file, so this notebook re-creates a stratified 80/10/10 split with seed 42, matching the split described on the tagasenti_model card as closely as possible. Your models' numbers here may differ by a fraction of a point from the card's reported 84.8 percent.
- **Label order of `dost-asti/RoBERTa-tl-sentiment-analysis`, `sdbrgo/...multiclass`, `scaredmeow/roberta...shopee` and `MENG21/...` is not documented in their configs.** The probe cell infers it from two sentences per sentiment and prints the inferred map, always double-check the printed map before citing numbers.
- **Binary models never predict neutral**, so their 3-class numbers would be unfair; they are only scored on binary sets where 3-class models compete by renormalizing negative vs positive.
- **`tabularisai` collapses Very Negative into Negative and Very Positive into Positive.**
- **The new sets add two different views.** `taglish_annotated_3class` has human-annotated neutral labels, which is the axis every model missed on the star-derived MTEB set. `lazada_stars_3class` is a second e-commerce platform with star-derived labels.
- **FiliSenti's TagaSenti score needs an overlap check.** Its model card says it trained on the full 35,686-row TagaSenti CSV and re-split 80/10/10. The benchmark's TagaSenti test is a different split of the same CSV, so a large share of the test rows were likely in its training pool. Treat the 0.9327 as in-domain until a row-level check proves otherwise.
- **Contamination map, now explicit:**
  - `raphgonda/FilipinoShopping` trained on ~400k Shopee reviews, so `shopee_binary`, `mteb_shopee_3class`, `taglish_annotated_3class` and `gadgets_train_probe` are all favorable territory for it.
  - `magixxixx/roberta-shopee-sentiment-gadgets` TRAINED on `gadgets_train_probe`, that cell is a train-accuracy check for them, zero-shot for everyone else.
  - `The-Thesis-Gods/lazada` trained on Lazada reviews, `lazada_stars_3class` is its home turf.
  - `letijo03/shopee-taglish-sentiment-analysis` (optional LoRA) trained on the file behind `taglish_annotated_3class`.
  - `tagasenti_test` and `hilisenti_test` remain the cleanest ground for the family: no competitor has published models trained on either. For FiliSenti, the cleanest column is Hiligaynon. For tagasenti_model, TagaSenti is clean.
  - `gadgets_train_probe` is a training-set check for magixxixx, not a scoring set.

## Optional: the LoRA adapter model

Run the cell below to also test `letijo03/shopee-taglish-sentiment-analysis`, an XLM-R base LoRA adapter. It needs the peft library and may fail if the adapter config is incomplete, which is why it is kept out of the main loop.

In [9]:
RUN_LORA = False  # flip to True to try the adapter model

if RUN_LORA:
    from peft import PeftModel
    base_id, lora_id = "xlm-roberta-base", "letijo03/shopee-taglish-sentiment-analysis"
    try:
        tok = AutoTokenizer.from_pretrained(base_id)
        model = AutoModelForSequenceClassification.from_pretrained(base_id, num_labels=2).to(DEVICE)
        model = PeftModel.from_pretrained(model, lora_id)
        # LoRA is binary. On 3-class sets, renormalize its 2 columns to neg/pos
        # (its config lists labels in neg/pos order, same convention as shopee_binary).
        eval_plan = [("shopee_binary", "binary"),
                     ("gadgets_train_probe", "binary"),
                     ("taglish_annotated_3class", "3class")]   # its home training file
        for dname, dkind in eval_plan:
            df, _ = datasets[dname]
            P = predict_probs(df["text"].tolist(), tok, model)
            P = P / P.sum(1, keepdims=True)
            pred = np.where(P.argmax(1) == 0, "negative", "positive")
            g = df["label"].values
            acc = accuracy_score(g, pred)
            f1m = f1_score(g, pred, average="macro", zero_division=0)
            tag = " (HOME TURF, trained on this file)" if dname == "taglish_annotated_3class" else ""
            print(f"letijo03 LoRA on {dname}{tag}: acc={acc:.4f} macroF1={f1m:.4f}")
    except Exception as e:
        print(f"LoRA eval failed: {e}")
    finally:
        try:
            del model, tok
        except NameError:
            pass
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()